In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ============================================================
# 0) Parameters (match SQL)
# ============================================================
tx_2024_start = "2024-01-01"
tx_2024_end   = "2024-12-31"
tx_2019_start = "2019-01-01"
tx_2019_end   = "2019-12-31"

asof_date = "2025-01-01"  # SQL uses this for Subscriber_Profile Last_Updated
recency_anchor = F.to_date(F.lit("2025-01-01"))

theatre_type = "Cineplex Theatre"

# ============================================================
# 1) Tables (your mappings)
# ============================================================
T_trans = spark.table("cpx_dataanalytics_gold.customer360.c360_f_transaction")
T_loc   = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_location")
T_sess  = spark.table("cpx_dataanalytics_gold.customer360.cpx_d_session")  # join optional; not used downstream
T_sub   = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscriber_profile_snapshot")
T_rfm   = spark.table("cpx_dataanalytics_gold.customer360.c360_f_rfm_model")

# ============================================================
# 2) tmp1: 2024 Theatre visits/revenue/tickets + recency (RFM base)
#    Mirrors first CTE tmp1 in SQL
# ============================================================
def safe_visit_date(col):
    return F.to_date(
        F.when(F.length(col.cast("string")) == 8, col.cast("string")),
        "yyyyMMdd"
    )


tx_base_2024 = (
    T_trans.alias("t")
    .join(T_loc.alias("l"), F.col("t.LocationId") == F.col("l.LocationID"), "left")
    .join(T_sess.alias("s"), F.col("t.SessionSkey") == F.col("s.sessionskey"), "left")
    .filter(F.col("t.CDE_ID").isNotNull())
    .filter(
    safe_visit_date(F.col("t.VisitDateId"))
    .between(tx_2024_start, tx_2024_end))

    .filter(F.col("l.LC_Location_Type_Description") == theatre_type)
    .filter((F.col("t.SCENEFLAG") == "Yes") | (F.col("t.CineClubFLAG") == "Yes"))
)

tmp1 = (
    tx_base_2024
    .groupBy("CDE_ID")
    .agg(
        # COUNT DISTINCT visits with Net_Revenue > 0
        F.countDistinct(F.when(F.col("Net_Revenue") > 0, F.col("CDE_Visit_ID"))).alias("No_of_Visits"),
        # COUNT DISTINCT visits with Net_Revenue > 0 AND Is_Concession=1
        F.countDistinct(
            F.when((F.col("Net_Revenue") > 0) & (F.col("ConcessionFLAG") == "Yes"), F.col("CDE_Visit_ID"))
        ).alias("Visits_with_Concession"),
        F.sum("Net_Revenue").alias("Net_Revenue"),
        # Regular vs premium ticket quantities (Auditorium_Experience_ID buckets)
        F.sum(
            F.when((F.col("Auditorium_Experience_ID").isin([1, 5, 6])) & (F.col("TicketFLAG") == "Yes"),
                   F.col("Quantity")).otherwise(F.lit(0))
        ).alias("No_of_Regular_Tickets"),
        F.sum(
            F.when((F.col("Auditorium_Experience_ID").isin([2, 3, 4, 7, 8])) & (F.col("TicketFLAG") == "Yes"),
                   F.col("Quantity")).otherwise(F.lit(0))
        ).alias("No_of_Premium_Tickets"),
        F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.col("Net_Revenue")).otherwise(F.lit(0))).alias("Net_Revenue_Concession"),
        F.sum(F.when(F.col("TicketFLAG") == "Yes", F.col("Net_Revenue")).otherwise(F.lit(0))).alias("Net_Revenue_Tickets"),
        # MAX Visit_Date for Recency
        F.max(safe_visit_date(F.col("VisitDateId"))).alias("Last_Visit_Date"),
    )
    .filter(F.col("Net_Revenue") > 0)  # HAVING SUM(Net_Revenue) > 0
    .withColumn(
        "Revenue_per_Visit",
        F.coalesce(F.col("Net_Revenue") / F.when(F.col("No_of_Visits") == 0, F.lit(None)).otherwise(F.col("No_of_Visits")), F.lit(0.0))
    )
    .withColumn("Recency", F.datediff(recency_anchor, F.col("Last_Visit_Date")))
    .drop("Last_Visit_Date")
)

# ============================================================
# 3) tmp -> CTE_B -> T1: RFM scoring + segment mapping
# ============================================================
tmp_rfm = (
    tmp1
    .withColumn(
        "R_score",
        F.when(F.col("Recency") >= 180, F.lit(1))
         .when(F.col("Recency") >= 90,  F.lit(2))
         .when(F.col("Recency") >= 45,  F.lit(3))
         .otherwise(F.lit(4))
    )
    .withColumn(
        "F_score",
        F.when(F.col("No_of_Visits") > 6, F.lit(4))
         .when(F.col("No_of_Visits") > 3, F.lit(3))
         .when(F.col("No_of_Visits") > 1, F.lit(2))
         .otherwise(F.lit(1))
    )
    .withColumn(
        "M_score",
        F.when(F.col("Revenue_per_Visit") >= 40, F.lit(4))
         .when(F.col("Revenue_per_Visit") >= 30, F.lit(3))
         .when(F.col("Revenue_per_Visit") >= 20, F.lit(2))
         .otherwise(F.lit(1))
    )
    .withColumn("RFM_Score", F.concat(F.col("R_score"), F.col("F_score"), F.col("M_score")).cast("int"))
)

T1 = (
    tmp_rfm
    .withColumn(
        "Segment",
        F.when(F.col("RFM_Score").isin([443, 444]), F.lit("Champions"))
         .when(F.col("RFM_Score").isin([232,233,234,242,243,244,332,333,334,342,343,344,432,433,434,442]), F.lit("Potential Loyalist"))
         .when(F.col("RFM_Score").isin([231,241,331,341,431,441]), F.lit("Value Conscious Loyals"))
         .when(F.col("RFM_Score").isin([221,222,223,224,321,322,323,324,421,422,423,424]), F.lit("Promising"))
         .when(F.col("RFM_Score").isin([311,312,313,314,411,412,413,414]), F.lit("New"))
         .when(F.col("RFM_Score").isin([121,122,123,124,131,132,133,134,141,142,143,144]), F.lit("At Risk"))
         .when(F.col("RFM_Score").isin([111,112,211,212,213,214]), F.lit("Fleeting"))
         .when(F.col("RFM_Score").isin([113,114]), F.lit("High Value Lapsed"))
         .otherwise(F.lit("NA"))
    )
)

# ============================================================
# 4) CTE_A: Attach recognitions (from subscriber snapshot as-of date) + ticket-type counts (member/companion)
# ============================================================
# --- Recognitions snapshot as-of (SQL: CAST(Last_Updated AS DATE) = '2025-01-01' AND Renewal_Status LIKE 'Active%')
# NOTE: adjust column names if your snapshot differs. These are best-guess based on your earlier usage.
sub_asof = (
    T_sub
    .withColumn("last_updated_date", F.to_date(F.col("Last_Updated")))
    .filter(F.col("last_updated_date") == F.lit(asof_date))
    .filter(F.col("Renewal_Status").like("Active%"))
    .groupBy("CDE_ID")
    .agg(
        F.sum("Recognitions_Redeemed").alias("Recognitions_Redeemed"),
        F.sum("Member_Priced_Recognitions_Redeemed").alias("Member_Priced_Recognitions_Redeemed"),
        F.sum("Recognitions_Left").alias("Recognitions_Left"),
    )
)

# --- Member vs companion ticket quantities (SQL uses CineClub_Discount_Type_ID 6=Member, 5=Companion)
t3_member_companion_2024 = (
    T_trans
    .filter((F.col("TicketFLAG") == "Yes") & (F.col("CineClubFLAG") == "Yes"))
    .filter(safe_visit_date(F.col("VisitDateId")).between(tx_2024_start, tx_2024_end))

    .groupBy("CDE_ID")
    .agg(
        F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 6, F.col("Quantity")).otherwise(F.lit(0))).alias("Member_Tickets"),
        F.sum(F.when(F.col("CineClub_Discount_Type_ID") == 5, F.col("Quantity")).otherwise(F.lit(0))).alias("Companion_Tickets"),
    )
)

cte_a = (
    T1
    .filter(F.col("Segment").isin(["Champions", "Potential Loyalist", "Value Conscious Loyals"]))
    .join(sub_asof, "CDE_ID", "left")
    .join(t3_member_companion_2024, "CDE_ID", "left")
)

cte_2024_rfm = (
    cte_a
    .groupBy(
        "CDE_ID",
        "Segment",
        F.when(F.coalesce(F.col("Recognitions_Redeemed"), F.lit(0)) > 0, F.lit("Yes")).otherwise(F.lit("No")).alias("Category"),
    )
    .agg(
        F.sum("No_of_Visits").alias("No_of_Visits_2024"),
        F.sum("Visits_with_Concession").alias("Visits_with_Concession_2024"),
        F.sum("Net_Revenue").alias("Net_Revenue_2024"),
        F.sum("Net_Revenue_Tickets").alias("Net_Revenue_Tickets_2024"),
        F.sum("Net_Revenue_Concession").alias("Net_Revenue_Concession_2024"),
        F.sum("No_of_Regular_Tickets").alias("No_of_Regular_Tickets_2024"),
        F.sum("No_of_Premium_Tickets").alias("No_of_Premium_Tickets_2024"),
        F.sum(F.coalesce(F.col("Member_Tickets"), F.lit(0))).alias("Member_Tickets"),
        F.sum(F.coalesce(F.col("Companion_Tickets"), F.lit(0))).alias("Companion_Tickets"),
    )
)

# ============================================================
# 5) CTE_2024 (SCENE-only) and CTE_2019 (SCENE-only) — mirrors SQL blocks named CTE_2023/CTE_2019
#    (Your SQL labels were "2023" but the date window is 2024; keeping year in names consistent with dates.)
# ============================================================
def build_scene_summary(start_date, end_date, suffix):
    base = (
        T_trans.alias("t")
        .join(T_loc.alias("l"), F.col("t.LocationId") == F.col("l.LocationID"), "left")
        .filter(safe_visit_date(F.col("t.VisitDateId")).between(start_date, end_date))
        .filter(F.col("l.LC_Location_Type_Description") == theatre_type)
        .filter(F.col("t.SCENEFLAG") == "Yes")   # <-- also fix: your flags are strings elsewhere
        .filter(F.col("t.CDE_ID").isNotNull())
    )

    # Prefix all metrics with scene_
    out = (
        base
        .groupBy("CDE_ID")
        .agg(
            F.countDistinct(F.when(F.col("Net_Revenue") > 0, F.col("CDE_Visit_ID"))).alias(f"scene_No_of_Visits_{suffix}"),
            F.countDistinct(F.when((F.col("Net_Revenue") > 0) & (F.col("ConcessionFLAG") == "Yes"), F.col("CDE_Visit_ID"))).alias(f"scene_Visits_with_Concession_{suffix}"),
            F.sum("Net_Revenue").alias(f"scene_Revenue_{suffix}"),
            F.sum(F.when(F.col("ConcessionFLAG") == "Yes", F.col("Net_Revenue")).otherwise(F.lit(0))).alias(f"scene_Rev_Conc_{suffix}"),
            F.sum(F.when(F.col("TicketFLAG") == "Yes", F.col("Net_Revenue")).otherwise(F.lit(0))).alias(f"scene_Rev_Tickets_{suffix}"),
            F.sum(F.when(F.col("TicketFLAG") == "Yes", F.col("Quantity")).otherwise(F.lit(0))).alias(f"scene_Tickets_{suffix}"),
        )
        .filter(F.col(f"scene_Revenue_{suffix}") > 0)
    )
    return out

cte_2024_scene = build_scene_summary(tx_2024_start, tx_2024_end, "2024")
cte_2019_scene = build_scene_summary(tx_2019_start, tx_2019_end, "2019")

# ============================================================
# 6) tmp5/tmp6 final rollup for Category='No' (non-redeemers) AND present in both 2019 & 2024 SCENE
# ============================================================
tmp5 = (
    cte_2024_rfm
    .filter(F.col("Category") == "No")
    .join(cte_2019_scene, on="CDE_ID", how="inner")
    .join(cte_2024_scene, on="CDE_ID", how="inner")
    .select(
        F.col("CDE_ID").alias("final_CDE_ID"),
        "Segment",
        # keep the RFM rollup (optional, but you only used Segment later)
        *[c for c in cte_2024_rfm.columns if c not in ["CDE_ID"]],
        # add SCENE metrics (now uniquely named)
        *[c for c in cte_2024_scene.columns if c != "CDE_ID"],
        *[c for c in cte_2019_scene.columns if c != "CDE_ID"],
    )
)

tmp6 = (
    tmp5
    .groupBy("final_CDE_ID")
    .agg(
        F.sum("scene_No_of_Visits_2024").alias("No_of_Visits_2024"),
        F.sum("scene_Visits_with_Concession_2024").alias("Visits_with_Concession_2024"),
        F.sum("scene_Revenue_2024").alias("Revenue_2024"),
        F.sum("scene_Rev_Conc_2024").alias("Rev_Conc_2024"),
        F.sum("scene_Rev_Tickets_2024").alias("Rev_Tickets_2024"),
        F.sum("scene_Tickets_2024").alias("Tickets_2024"),

        F.sum("scene_No_of_Visits_2019").alias("No_of_Visits_2019"),
        F.sum("scene_Visits_with_Concession_2019").alias("Visits_with_Concession_2019"),
        F.sum("scene_Revenue_2019").alias("Revenue_2019"),
        F.sum("scene_Rev_Conc_2019").alias("Rev_Conc_2019"),
        F.sum("scene_Rev_Tickets_2019").alias("Rev_Tickets_2019"),
        F.sum("scene_Tickets_2019").alias("Tickets_2019"),
    )
)


final_summary = (
    tmp6
    .agg(
        F.count("final_CDE_ID").alias("No_of_members"),
        F.sum("No_of_Visits_2024").alias("No_of_Visits_2024"),
        F.sum("Visits_with_Concession_2024").alias("Visits_with_Concession_2024"),
        F.sum("Revenue_2024").alias("Revenue_2024"),
        F.sum("Rev_Conc_2024").alias("Rev_Conc_2024"),
        F.sum("Rev_Tickets_2024").alias("Rev_Tickets_2024"),
        F.sum("Tickets_2024").alias("Tickets_2024"),

        F.sum("No_of_Visits_2019").alias("No_of_Visits_2019"),
        F.sum("Visits_with_Concession_2019").alias("Visits_with_Concession_2019"),
        F.sum("Revenue_2019").alias("Revenue_2019"),
        F.sum("Rev_Conc_2019").alias("Rev_Conc_2019"),
        F.sum("Rev_Tickets_2019").alias("Rev_Tickets_2019"),
        F.sum("Tickets_2019").alias("Tickets_2019"),
    )
)


# display(final_summary)

# ============================================================
# 7) "tmp" block from SQL (RFM_Model membership present; exclude active cineclub as-of)
#    Included here for completeness; not used in final_summary above in your SQL excerpt.
# ============================================================
active_cineclub_asof = (
    T_sub
    .withColumn("last_updated_date", F.to_date(F.col("Last_Updated")))
    .filter(F.col("last_updated_date") == F.lit(asof_date))
    .filter(F.col("Renewal_Status").like("Active%"))
    .select("CDE_ID").distinct()
)

tmp_membership_filtered = (
    cte_2024_scene.alias("a")
    .join(cte_2019_scene.select("CDE_ID"), "CDE_ID", "inner")
    .join(T_rfm.select("CDE_ID", "SCENE_Membership_ID").alias("r"), "CDE_ID", "left")
    .filter(F.col("r.SCENE_Membership_ID").isNotNull())
    .join(active_cineclub_asof.alias("x"), "CDE_ID", "left_anti")  # NOT IN active cineclub
)

# ============================================================
# 8) Samplers block (requires event-grain from mkt_subscriber_profile_snapshot)
#    Assumptions: columns exist in T_sub:
#      - CC_User_Profile_ID, CDE_ID, Invoice_Date, Event_Type, Plan_ID
#    If your real column names differ, rename them here.
# ============================================================
T_events = spark.table("cpx_dataanalytics_gold.marketing.mkt_subscription_events")

# 1) Create a stable mapping: one row per cc_user_profile_id -> CDE_ID
#    (pick the latest available CDE_ID for that profile_id)
w_map = Window.partitionBy("cc_user_profile_id").orderBy(F.col("snapshot_date_id").desc())

sub_map = (
    T_sub
    .select("cc_user_profile_id", "cde_id", "snapshot_date_id")
    .filter(F.col("cc_user_profile_id").isNotNull())
    .filter(F.col("cde_id").isNotNull())
    .withColumn("rn", F.row_number().over(w_map))
    .filter(F.col("rn") == 1)
    .select("cc_user_profile_id", "cde_id")
)

# 2) Filter events for 2024 monthly1 (apply plan_id filter on EVENTS, not snapshot)
events_2024_monthly = (
    T_events
    .filter(F.col("reporting_datetime").isNotNull())
    .filter(F.year(F.to_timestamp("reporting_datetime")) == 2024)
    .filter(F.col("plan_id") == "monthly1")
    .select("cc_user_profile_id", "event_type")
)

# 3) Aggregate to sampler metrics at CDE_ID + cc_user_profile_id grain (matches SQL)
cte_cc = (
    events_2024_monthly
    .join(sub_map, on="cc_user_profile_id", how="inner")
    .groupBy("cde_id", "cc_user_profile_id")
    .agg(
        F.sum(F.when(F.col("event_type") == "subscription_created", 1).otherwise(0)).alias("No_of_times_created"),
        F.sum(F.when(F.col("event_type") == "subscription_renewed", 1).otherwise(0)).alias("No_of_times_renewed"),
    )
)

cte_cc1 = (
    cte_cc
    .withColumn(
        "Cineclub_Category",
        F.when(F.col("No_of_times_created") >= 2, "Samplers")
         .when((F.col("No_of_times_created") == 1) & (F.col("No_of_times_renewed") == 0), "One time subscribers")
         .when((F.col("No_of_times_created") == 1) & (F.col("No_of_times_renewed").between(1, 4)), "Seasonal goers")
         .otherwise("Program lovers")
    )
)

# 4) Join back to RFM table using consistent key casing (CDE_ID vs cde_id)
samplers_output = (
    cte_2024_rfm
    .filter(F.col("Category") == "Yes")
    .join(cte_cc1.select(F.col("cde_id").alias("CDE_ID"), "Cineclub_Category"), on="CDE_ID", how="inner")
)


In [0]:
display(final_summary)

In [0]:
display(samplers_output)

In [0]:
# 1) How many "Category=Yes" members exist in your RFM population?
cte_2024_rfm.groupBy("Category").count().show()

# 2) How many events survive your 2024 + monthly1 filtering?
events_2024_monthly.groupBy("plan_id").count().show()
events_2024_monthly.groupBy("event_type").count().show()

print("events_2024_monthly rows:", events_2024_monthly.count())
print("sub_map rows:", sub_map.count())
print("cte_cc1 rows:", cte_cc1.count())
print("cte_cc1 distinct cde_id:", cte_cc1.select("cde_id").distinct().count())
print("cte_2024_rfm distinct CDE_ID:", cte_2024_rfm.select("CDE_ID").distinct().count())


In [0]:
rfm_ids = cte_2024_rfm.filter(F.col("Category") == "Yes").select("CDE_ID").distinct()
sampler_ids = cte_cc1.select(F.col("cde_id").alias("CDE_ID")).distinct()

print("Overlap count:",
      sampler_ids.join(rfm_ids, "CDE_ID", "inner").count())

# show a few overlapping IDs if any exist
sampler_ids.join(rfm_ids, "CDE_ID", "inner").show(20, False)
